In [ ]:
# ==============================================================================
# SETUP
# ==============================================================================
from helpers import IncrementalPipeline, TableConfig, setup_logger
from datetime import datetime

# Setup logging
logger = setup_logger("incremental_facts")

# Initialize pipeline
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

logger.info("Incremental fact load initialized")

In [ ]:
# ==============================================================================
# SILVER TRANSFORMATION FUNCTIONS (Imported from shared module)
# ==============================================================================
from helpers.silver_transforms import (
    transform_service_fact,
    transform_rental_fact
)

logger.info("Silver transformation functions imported from shared module")

In [ ]:
# ==============================================================================
# PRE-LOAD DEPENDENCY TABLES (Required for fact transformations)
# ==============================================================================

# Load dependency tables to bronze (no transformations needed)
dependency_tables = ["staff", "inventory", "payment"]

logger.info(f"Pre-loading {len(dependency_tables)} dependency tables to bronze")

for table in dependency_tables:
    try:
        logger.info(f"Loading dependency: {table}")
        df = pipeline.bronze_loader.load_incremental(table_name=table, watermark_column="last_update", force_full=False)
        row_count = df.count()

        # Merge to bronze (create if not exists)
        pipeline.bronze_loader.merge_to_bronze(
            df=df,
            table_name=table,
            business_key=f"{table}_id"
        )

        # Update watermark
        load_type = "FULL" if not pipeline.bronze_loader.watermark_manager.has_watermark(table) else "INCREMENTAL"
        pipeline.bronze_loader.update_watermark(
            table_name=table,
            df=df,
            watermark_column="last_update",
            load_type=load_type
        )

        logger.info(f"✅ {table}: {row_count:,} rows loaded")
    except Exception as e:
        logger.error(f"❌ Failed to load {table}: {str(e)}")
        raise

logger.info("All dependency tables loaded to bronze")

In [ ]:
# ==============================================================================
# FACT TABLE CONFIGURATIONS (DRY - Single Source of Truth)
# ==============================================================================

fact_configs = [
    TableConfig(
        table_name="service",
        business_key="service_id",
        surrogate_key="service_key",
        watermark_column="service_date",
        scd_type=1,  # Facts use SCD1 logic with append mode
        gold_table_name="fact_service",
        silver_transform=transform_service_fact
    ),
    TableConfig(
        table_name="rental",
        business_key="rental_id",
        surrogate_key="rental_key",
        watermark_column="rental_date",
        scd_type=1,  # Facts use SCD1 logic with append mode
        gold_table_name="fact_rental",
        silver_transform=transform_rental_fact,
        dependencies=["staff", "inventory", "payment"]
    ),
]

logger.info(f"Configured {len(fact_configs)} fact tables")

In [ ]:
# ==============================================================================
# EXECUTE INCREMENTAL LOAD (DRY - Single Function Call)
# ==============================================================================

# Load all facts incrementally
results = pipeline.load_tables(fact_configs, force_full=False)

# Display results
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Fact table row counts
print("Fact Table Row Counts:")
print(f"fact_service: {spark.table('wheelie.gold.fact_service').count():,}")
print(f"fact_rental: {spark.table('wheelie.gold.fact_rental').count():,}")

# Check latest watermarks
print("\nLatest Watermarks:")
display(spark.table("wheelie.monitoring.watermarks"))